In [1]:
import argparse, asyncio, yaml, random, logging, time, os

from typing import Union

#from concurrent.futures import ThreadPoolExecutor
#from radical.asyncflow import LocalExecutionBackend
#from rhapsody.backends import DragonExecutionBackendV3

#from dragon.infrastructure.policy import Policy
#from dragon.native.machine import System, Node
#from dragon.data.ddict import DDict

#from radical.asyncflow import WorkflowEngine
#from radical.asyncflow.logging import init_default_logger

import multiprocessing as mp

In [ ]:
foldseek_path = "/work/hdd/bdyk/apark4/foldseek/bin/foldseek"
colabfold_path = "/work/nvme/bdyk/apark4/localcolabfold/.pixi/envs/default/bin/colabfold_batch"
runfold_path = "/work/nvme/bdyk/apark4/ROME/run_fold.sh"
import os
os.environ["HF_TOKEN"] = ""
os.environ["HF_HOME"] = "/work/nvme/bdyk/apark4/huggingface"
os.environ["MPLBACKEND"]="notebook"
os.environ["CUDA_LAUNCH_BLOCKIN"]="1"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["TRL_EXPERIMENTAL_SILENCE"]="1"
foldseekdb_path = "/work/hdd/bdyk/apark4/foldseek/afdb50"
afdb_name= "afdb50"

superfamilies = [
    ("chey", "CheY-like superfamily"),
    ("tphd", "Tetratricopeptide-like helical domain superfamily"),
    ("sammt", "S-adenosyl-L-methionine-dependent methyltransferase superfamily"),
    ("trx", "Thioredoxin-like superfamily"),
    ("cnh", "Carbon-nitrogen hydrolase superfamily"),
    ("whdbd","Winged helix-like DNA-binding domain superfamily"),
    ("mccd", "Mitochondrial carrier domain superfamily"),
    ("vdcd", "Voltage-dependent channel domain superfamily"),
]


In [3]:
def generate_seq_epgf(superfamily, num_seq=64, gen_fam_ddict=None, seq_ddict=None):
    import os
    import re
    import math
    import random
    import numpy as np
    import torch
    from ProteinModel import ProteinSequenceScorer
    from transformers import AutoTokenizer, LlamaForCausalLM, GenerationConfig, AutoModelForCausalLM
    from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig
    from tqdm import tqdm
    import multiprocessing as mp
    print(torch.cuda.device_count())
    if gen_fam_ddict is not None:
        print("DDict passed to generate")
    def set_seed(seed: int = 42) -> None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        np.random.seed(seed)
        random.seed(seed)
    def load_llama_or_latest_checkpoint(
        base_model_id: str,
        lora_id: str,
        dtype=torch.bfloat16,
        device_map="auto",
    ):
        """
        Load base model and optionally attach a local LoRA adapter (directory).
        If `lora_id` is a directory containing a checkpoint, attach it via
        `PeftModel.from_pretrained`. Otherwise, if `args.apply_lora` is set,
        construct a `LoraConfig` and wrap the base model with `get_peft_model`.
    
        Returns: model, tokenizer, loaded_from, iteration
        """
        last_checkpoint = None
        from pathlib import Path
        if (lora_id is not None) and os.path.isdir(lora_id) and any(Path(lora_id).iterdir()):
            last_checkpoint = True
    
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", padding_side="left", use_fast=True)
        tokenizer.pad_token = tokenizer.eos_token
        iteration = 0
        if last_checkpoint is not None:
            print(f"Found LoRA checkpoint")
            print(f"Loading base model: {base_model_id}")
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                dtype=dtype,
                device_map=device_map,
            )
            # Attach LoRA adapter weights
            print("Applying LoRA adapter from checkpoint...")
            model = PeftModel.from_pretrained(base_model, lora_id, is_trainable=True)
            loaded_from = last_checkpoint
        else:
            print(f"No checkpoint found, loading base model: {base_model_id}")
            lora_config = LoraConfig(
                r=128,
                lora_alpha=256,
                lora_dropout=0.05,
                inference_mode=False,
                bias="none",
                task_type="CAUSAL_LM",
            )
            model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                dtype=dtype,
                device_map=device_map,
            )
            model = get_peft_model(model, lora_config)
            loaded_from = base_model_id
    
        return model, tokenizer, loaded_from
    
    model_path = "GreatCaptainNemo/ProLLaMA"
    lora_id = "prolora"
    model, tokenizer, loaded_from = load_llama_or_latest_checkpoint(
        model_path,
        lora_id,
    )
    generation_config = GenerationConfig(
        max_new_tokens=1,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=1,
        num_return_sequences=8,
        repetition_penalty=1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    def softmax(x, temperature=1.0):
        x = np.array(x) / temperature
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    initial_temperature = 1.0
    final_temperature = 0.001
    decay_rate = 0.1
    
    # maximum allowed sequence length (chars / tokens)
    MAX_SEQ_LEN = 500
    
    model.eval()
    
    seq_with_scores = []
    set_seed()
    with torch.no_grad():
        i = 0
        #gennerate num_seq protein sequences
        while i < num_seq:
            temperature = initial_temperature
            failed = False
            prompt = f'[Generate by superfamily] Superfamily=<{superfamily}> Seq=<'  #you can modify this prompt
            original_prompt_id = tokenizer(prompt, return_tensors="pt").input_ids[0].tolist()
            logp = []
            tokens = []
            while True:
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                outputs = model.generate(
                    **inputs,
                    generation_config=generation_config,
                    output_scores=True,
                    return_dict_in_generate=True
                )
    
                candidates = tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)
                transition_scores = model.compute_transition_scores(outputs.sequences, outputs.scores, normalize_logits=True)
                log_probs = transition_scores.sum(dim=1).cpu().numpy().tolist() # log prob used as score
    
                candidates_with_scores = [(candidates[idx], outputs.sequences[idx], log_probs[idx]) for idx in range(len(candidates))]
                candidates_with_scores.sort(key=lambda x: x[2], reverse=True)
    
                top_half_count = max(1, len(candidates) // 2)
                top_candidates = candidates_with_scores[:top_half_count]
    
                for cand, tok, score in candidates_with_scores:
                    if cand.strip().endswith('>') and not any(c == cand and s == score for c, _, s in top_candidates):
                        top_candidates.append((cand, tok, score))
    
                bio_scores = []
                filtered_candidates = []
                filtered_cand_logp  = []
                filtered_cand_tokens= []
    
                for cand, token, lm_score in top_candidates:
                    seq = cand.split('Seq=<')[-1].split('>')[0].strip()
                    # skip sequences longer than MAX_SEQ_LEN characters or tokens
                    if len(token) > MAX_SEQ_LEN:
                        continue
                    scorer = ProteinSequenceScorer(seq)
                    bio_score = scorer.get_comprehensive_score()
    
                    if bio_score < 0.55:
                        continue
                    bio_scores.append(bio_score)
                    filtered_candidates.append(cand)
                    filtered_cand_logp.append(lm_score)
                    filtered_cand_tokens.append(token)
                if len(filtered_candidates) == 0:
                    failed = True
                    break
    
                bio_scores_softmax = softmax(bio_scores, temperature=temperature)
                temperature = max(final_temperature, temperature * decay_rate)
                sampled_idx = np.random.choice(len(filtered_candidates), p=bio_scores_softmax)
                winner = filtered_candidates[sampled_idx]
                winner_logp = filtered_cand_logp[sampled_idx]
                winner_token = filtered_cand_tokens[sampled_idx][-1]
                if winner.endswith('>'):
                    logp.append(winner_logp)
                    tokens.append(winner_token.item())
                    break
                else:
                    prompt = winner
                    logp.append(winner_logp)
                    tokens.append(winner_token.item())
    
            if not failed:
                sequence = winner.split('Seq=<')[-1].split('>')[0].strip()
                seq_with_scores.append({"prompt_id": original_prompt_id, "sequence":sequence, "tokens":tokens, "logp":logp, "superfamily": superfamily})
                if gen_fam_ddict is not None:
                    print("adding to ddict")
                    #gen_fam_ddict[sequence] = {"prompt_id": original_prompt_id, "tokens":tokens, "logp":logp, "superfamily": superfamily}
                    if superfamily not in gen_fam_ddict:
                        gen_fam_ddict[superfamily] = []
                    gen_fam_ddict[superfamily].append({"sequence": sequence, "prompt_id": original_prompt_id, "tokens":tokens, "logp":logp})
                if seq_ddict is not None:
                    seq_ddict[sequence] = {"prompt_id": original_prompt_id, "tokens":tokens, "logp":logp, "superfamily": superfamily}
                i += 1
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return seq_with_scores

In [4]:
gen_fam_dict={}
seq_dict={}
result = generate_seq_epgf("CheY-like superfamily", num_seq=4, gen_fam_ddict=gen_fam_dict, seq_ddict=seq_dict)

1
DDict passed to generate
No checkpoint found, loading base model: GreatCaptainNemo/ProLLaMA


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/u/apark4/ve/test_venv/lib/python3.11/site-packages/Bio/SeqUtils/ProtParam.py:106: BiopythonDeprecationWarning: The get_amino_acids_percent method has been deprecated and will likely be removed from Biopython in the near future. Please use the amino_acids_percent attribute instead.
  warnings.warn(


adding to ddict
adding to ddict
adding to ddict
adding to ddict


In [5]:
def fold_sequences(generated_families_ddict, superfamily, staging_dir, output_dir, finished_family_ddict, seed=None):
    import os, subprocess, re, shutil
    from pathlib import Path
    if seed is None:
        import random
        seed = random.randint(0, 65535)
    
    output_path = Path(output_dir)
    staging_path = Path(staging_dir)
    
    output_path.mkdir(parents=True, exist_ok=True)
    staging_path.mkdir(parents=True, exist_ok=True)
    superfamily_sequences = generated_families_ddict[superfamily]

    input_file = staging_path / "sequences.fasta"
    with open(input_file, 'w') as f:
        for i, seq_info in enumerate(superfamily_sequences):
            sequence = seq_info["sequence"]
            f.write(f'>{i}\n')
            f.write(sequence + '\n')
        #for i, seq in enumerate(sequences):
        #    f.write(f'>{i}\n')
        #    f.write(seq + '\n')
    
    #colabfold_result = subprocess.run([
    #    colabfold_path,
    #    "--num-recycle", "3",
    #    "--templates",
    #    "--num-models",  "5",
    #    "--model-order", "1,2,3,4,5",
    #    "--random-seed", f"{seed}",
    #    input_file,
    #    staging_dir
    #], capture_output=True, text=True)
    
    colabfold_result = subprocess.run([
        runfold_path,
        input_file,
        staging_dir
    ], capture_output=True, text=True)
    print(colabfold_result.stdout)
    
    #staging_path = Path(staging_dir)
    #staging_path.mkdir(parents=True, exist_ok=True)

    pattern = re.compile(r'^(.+)_unrelaxed_rank_001_alphafold2_ptm_model_\d+_seed_000\.pdb$')

    for pdb_file in staging_path.glob('*.pdb'):
        if pattern.match(pdb_file.name):
            shutil.copy(pdb_file, output_path / pdb_file.name)
    finished_family_ddict[superfamily] = len(superfamily_sequences)
    return superfamily

In [6]:
finished_family_dict={}
fold_res = fold_sequences(gen_fam_dict, "CheY-like superfamily", "/work/nvme/bdyk/apark4/ROME/stage", "/work/nvme/bdyk/apark4/ROME/output", finished_family_dict)

In [7]:
def score_sequence(superfamily, pdb_input_dir, db, tmp_dir, output_file, scored_ddict, seq_ddict = None, format_output='query,target,lddt,prob'):
    # Perform foldseek search on sequences and return hprob scores
    import subprocess
    import csv
    from pathlib import Path
    import re
    if seq_ddict is not None:
        print("DDict passed to fold")
    pdb_input_path = Path(pdb_input_dir)
    tmp_path       = Path(tmp_dir)
    output_path    = Path(output_file)
    tmp_path.mkdir(parents=True, exist_ok=True)
    pdb_count = len(list(pdb_input_path.glob('*.pdb')))
    # Run foldseek
    cmd = [
        foldseek_path, 'easy-search',
        str(pdb_input_path),
        db,
        str(output_path),
        str(tmp_path),
        '--format-output', format_output,
    ]
    #result = subprocess.run(cmd, capture_output=True, text=True)
    #if result.returncode != 0:
    #    raise RuntimeError(f"Foldseek failed:\n{result.stderr}")
    
    def load_fasta_sequences(fasta_path: str) -> list[str]:
        sequences = []
        current = []
        with open(fasta_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if current:
                        sequences.append("".join(current))
                        current = []
                else:
                    current.append(line)
            if current:
                sequences.append("".join(current))
        return sequences
    
    # Parse foldseek output and extract hprob scores
    def parse_sequence_id(query_name):
        m = re.match(r'^(\d+)_unrelaxed_rank_001', query_name)
        return m.group(1) if m else query_name

    hits = {}
    with open(output_path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 4:
                continue
            query, target, lddt, prob = parts[0], parts[1], float(parts[2]), float(parts[3])
            seq_id = str(parse_sequence_id(query))
            if seq_id not in hits:
                hits[seq_id]=[]
            hits[seq_id].append((target, lddt, prob))
    for i in range(pdb_count):
        if str(i) not in hits:
            hits[str(i)].append(("None", 0, 0))
    # create scored sequence list with top hit and average scores
    sequences = load_fasta_sequences("/work/nvme/bdyk/apark4/ROME/stage/sequences.fasta")
    scored_sequences = []
    for seq_id, matches in sorted(hits.items(), key=lambda x: int(x[0]) if x[0].isdigit() else x[0]):
        total = len(matches)
        top_target, top_lddt, top_hscore = matches[0]  # first hit is top match
        avg_lddt   = sum(m[1] for m in matches) / total
        avg_prob = sum(m[2] for m in matches) / total
        seq = sequences[int(seq_id)]
        scored_sequences.append({
            "sequence":   seq,
            "total_hits": total,
            "top_target": top_target,
            "top_lddt":   top_lddt,
            "top_prob":   top_hscore,
            "avg_lddt":   avg_lddt,
            "avg_prob":   avg_prob,
        })
        scored_ddict[seq] = {
            "total_hits": total,
            "top_target": top_target,
            "top_lddt":   top_lddt,
            "top_prob":   top_hscore,
            "avg_lddt":   avg_lddt,
            "avg_prob":   avg_prob,
        }
        if seq_ddict is not None:
            # get existing ddict entry for the sequence
            existing_entry = seq_ddict.get(seq, {})
            # update the entry with new foldseek scores
            existing_entry.update({
                "total_hits": len(matches),
                "top_target": top_target,
                "top_lddt":   top_lddt,
                "top_prob":   top_hscore,
                "avg_lddt":   avg_lddt,
                "avg_prob":   avg_prob,
            })
            seq_ddict[seq] = existing_entry

    return scored_sequences

In [8]:
scored_dict={}
score_res = score_sequence("CheY-like superfamily", "/work/nvme/bdyk/apark4/ROME/output", "/work/hdd/bdyk/apark4/foldseek/afdb50/chey", "/work/nvme/bdyk/apark4/ROME/tmp", "/work/nvme/bdyk/apark4/ROME/output/score.txt", scored_dict, seq_dict)
#def score_sequence(superfamily, pdb_input_dir, db, tmp_dir, output_file, scored_ddict, seq_ddict = None, format_output='query,target,lddt,prob'):


DDict passed to fold


In [9]:
print(seq_dict)

{'MLTNQKILIVDDDPGFRELLKLALKKAGYEVVSAKDGQEALKLFKEKKPDLVITDWMMPGMDGYEVCRKIRSHPKLKNIPVIMVTAYAQVGDREKALAAGADDYLVKPFELEELKARVRKVLGKGK': {'prompt_id': [1, 518, 5631, 403, 491, 2428, 11922, 29962, 5670, 11922, 29922, 29966, 26856, 29979, 29899, 4561, 2428, 11922, 29958, 25981, 29922, 29966], 'tokens': [1988, 29911, 29940, 29984, 29968, 29902, 5265, 29963, 7858, 11191, 29954, 29943, 1525, 2208, 29968, 29931, 1964, 29968, 29968, 10051, 29979, 22240, 29963, 8132, 29968, 29928, 29954, 29984, 29923, 1964, 29968, 29931, 29943, 6059, 29968, 29968, 29925, 19558, 29963, 1806, 29928, 26735, 3580, 29954, 5773, 29954, 29979, 29923, 8257, 29934, 29968, 8193, 29903, 3954, 29968, 29931, 29968, 29940, 5690, 29963, 7833, 29963, 6040, 29979, 29909, 29984, 29963, 29954, 29928, 1525, 29968, 1964, 6344, 29954, 17744, 29979, 29931, 29963, 29968, 13691, 29923, 1307, 6670, 29968, 1718, 29963, 29934, 29968, 29963, 29931, 29954, 29968, 29954, 29968, 29958], 'logp': [-2.7654991149902344, -2.7970385551452637, -2.6774921

In [10]:
def grpo_trainer(superfamily_ddict, gen_fam_ddict, scored_ddict):
    import logging, os, re
    import os

    # Force single-node single-process distributed setup
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    os.environ["RANK"] = "0"
    os.environ["LOCAL_RANK"] = "0"
    os.environ["WORLD_SIZE"] = "1"
    
    # Unset SLURM variables that confuse accelerate into thinking multinode
    os.environ.pop("SLURM_PROCID", None)
    os.environ.pop("SLURM_NODEID", None)
    os.environ.pop("SLURM_NTASKS", None)
    os.environ.pop("SLURM_NPROCS", None)
    os.environ.pop("SLURM_STEP_NODELIST", None)
    os.environ.pop("SLURM_JOB_NODELIST", None)
    import numpy as np
    import torch
    from transformers import AutoTokenizer, LlamaForCausalLM, GenerationConfig, AutoModelForCausalLM
    from datasets import Dataset
    from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig
    from trl import GRPOConfig, GRPOTrainer

    def load_llama_or_latest_checkpoint(
        base_model_id: str,
        lora_id: str,
        dtype=torch.bfloat16,
        device_map="auto",
    ):
        """
        Load base model and optionally attach a local LoRA adapter (directory).
        If `lora_id` is a directory containing a checkpoint, attach it via
        `PeftModel.from_pretrained`. Otherwise, if `args.apply_lora` is set,
        construct a `LoraConfig` and wrap the base model with `get_peft_model`.
    
        Returns: model, tokenizer, loaded_from, iteration
        """
        from pathlib import Path
        last_checkpoint = None
    
        if lora_id and os.path.isdir(lora_id) and any(Path(lora_id).iterdir()):
            last_checkpoint = True
    
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf", padding_side="left", use_fast=True)
        tokenizer.pad_token = tokenizer.eos_token
        iteration = 0
        if last_checkpoint is not None:
            print(f"Found LoRA checkpoint")
            print(f"Loading base model: {base_model_id}")
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                dtype=dtype,
                device_map=device_map,
            )
            # Attach LoRA adapter weights
            print("Applying LoRA adapter from checkpoint...")
            model = PeftModel.from_pretrained(base_model, lora_id, is_trainable=True)
            loaded_from = last_checkpoint
        else:
            print(f"No checkpoint found, loading base model: {base_model_id}")
            lora_config = LoraConfig(
                r=128,
                lora_alpha=256,
                lora_dropout=0.05,
                inference_mode=False,
                bias="none",
                task_type="CAUSAL_LM",
            )
            model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                dtype=dtype,
                device_map=device_map,
            )
            model = get_peft_model(model, lora_config)
            loaded_from = base_model_id
    
        return model, tokenizer, loaded_from
    max_prompt_length=2048
    max_seq_length=512
    model_path = "GreatCaptainNemo/ProLLaMA"
    lora_id = "prolora"
    model, tokenizer, loaded_from = load_llama_or_latest_checkpoint(
        model_path,
        lora_id,
    )
    training_args = GRPOConfig(
        learning_rate = 5e-6,
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        logging_steps = 1,
        per_device_train_batch_size = 4, # how many to process at once per gpu
        gradient_accumulation_steps = 1, # how many steps to accumulate
        num_generations = 4, # How many generations for each prompt
        generation_batch_size = 4,
        max_completion_length = 1024,
        max_steps = 1,
        #save_steps = 4,
        max_grad_norm = 1.0,
        report_to = "none", # Can use Weights & Biases
        run_name=f"prolora-rome",
        output_dir = lora_id,
        overwrite_output_dir=True,
    )
    # each process receives: per_device_train_batch_size unique prompts
    # each process must return: per_device_train_batch_size × num_generations completions
    def rollout_func(prompts: list[str], trainer: GRPOTrainer, output_ddict=superfamily_ddict, gen_fam_ddict=gen_fam_ddict):
        #model = trainer.model
        #tokenizer = trainer.processing_class
        #inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        eos_id = trainer.processing_class.eos_token_id
        device = trainer.model.device
        import time
        n_gen = 4
        total_gens = len(prompts)
        
        for prompt in list(set(prompts)):
            num = prompts.count(prompt)
            output_ddict[prompt] = num
            
        # wait for generations to be done
        prompt_ids, completion_ids, logprobs = [], [], []
        print(prompts)
        for prompt in list(set(prompts)):
            p_gens = len(gen_fam_ddict.get(prompt, []))
            while p_gens < n_gen:
                print(f"Prompt {prompt} has {p_gens}/{n_gen} generations completed. Waiting...")
                time.sleep(1)
                p_gens = len(gen_fam_ddict.get(prompt, []))
            gens_for_prompt = gen_fam_ddict[prompt]
            for gen_result in gens_for_prompt:
                original_prompt_id = gen_result["prompt_id"]
                sequence = gen_result["sequence"]
                tokens = gen_result["tokens"] + [eos_id]
                logp = gen_result["logp"] + [0.0]

                prompt_id = original_prompt_id
                completion_id = tokens
                logprob = logp

                prompt_ids.append(prompt_id)
                completion_ids.append(completion_id)
                logprobs.append(logprob)
        print("out of rollout")
        print({
            "prompt_ids": len(prompt_ids),
            "completion_ids": len(completion_ids),
            "logprobs": len(logprobs),
        })
        return {
            "prompt_ids": prompt_ids,
            "completion_ids": completion_ids,
            "logprobs": logprobs,
        }

    def sequence_reward(prompts, completions, seq_ddict=scored_ddict, **kwargs):
        rewards = []
        import time
        for prompt, completion in zip(prompts, completions):
            seq = completion[:-1] if completion.endswith('>') else completion
            while seq not in seq_ddict.keys():
                print(f"Sequence {seq} not scored yet. Waiting...")
                time.sleep(1)
            seq_scores = seq_ddict[seq]
            reward = seq_scores["top_prob"]  # use average foldseek probability as reward signal
            print(reward)
            rewards.append(reward)
            
            #rewards.append(1.0)
        print("out of reward")
        return rewards
    
    # build dataset of prompts from list of sequences
    # dataset is just list of prompts, it is raw superfamily string, rollout_func will handle prompt formatting
    formatted_data = [{"prompt": "CheY-like superfamily"}]
    superfamily_dataset = Dataset.from_list(formatted_data)
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        rollout_func=rollout_func,
        reward_funcs = [
            sequence_reward
        ],
        args = training_args,
        train_dataset = superfamily_dataset,
    )
    #trainer.train()
    return trainer

In [11]:
import trl.trainer.utils as trl_utils
_orig_shuffle = trl_utils.shuffle_sequence_dict

def debug_shuffle(seq_dict):
    import torch
    print("=== shuffle_sequence_dict called ===")
    for k, v in seq_dict.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: tensor {v.shape}")
        elif isinstance(v, list):
            print(f"  {k}: list len={len(v)}")
    return _orig_shuffle(seq_dict)

trl_utils.shuffle_sequence_dict = debug_shuffle

In [12]:
superfamily_dict={}
trainer = grpo_trainer(superfamily_dict, gen_fam_dict, seq_dict)

No checkpoint found, loading base model: GreatCaptainNemo/ProLLaMA


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [13]:
training_args=trainer.args
print(training_args.generation_batch_size)
print(training_args.per_device_train_batch_size)
print(training_args.num_generations)

4
4
4


In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


['CheY-like superfamily', 'CheY-like superfamily', 'CheY-like superfamily', 'CheY-like superfamily']
out of rollout
{'prompt_ids': 4, 'completion_ids': 4, 'logprobs': 4}
1.0
1.0
1.0
1.0
out of reward
=== shuffle_sequence_dict called ===
  prompt_ids: tensor torch.Size([4, 22])
  prompt_mask: tensor torch.Size([4, 22])
  completion_ids: tensor torch.Size([4, 158])
  completion_mask: tensor torch.Size([4, 158])
  advantages: tensor torch.Size([4])
  num_items_in_batch: tensor torch.Size([])
  sampling_per_token_logps: tensor torch.Size([4, 158])


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.000000


TrainOutput(global_step=1, training_loss=0.0, metrics={'train_runtime': 1.5274, 'train_samples_per_second': 2.619, 'train_steps_per_second': 0.655, 'total_flos': 0.0, 'train_loss': 0.0})

In [ ]:
async def gen_seq_listen_launch(gen_fam_ddict, seq_ddict, terminate_event):
    generated_families = []
    gen_task_futures = []
    while not terminate_event.is_set():
        superfam_to_gen = []
        task_to_launch = []
        if shared_ddict:
            for family, num in gen_fam_ddict.items():
                if family not in generated_families:
                    superfam_to_gen.append((family, num))
            for family, num in superfam_to_gen:
                gpu_alloc = gpu_allocator.next()
                task_policy_rr = Policy(
                    placement=Policy.Placement.HOST_NAME,
                    host_name=gpu_alloc[0],
                    gpu_affinity=gpu_alloc[1],
                )
                task_fut = generate_seq_epgf(family, num, gen_fam_ddict=gen_fam_ddict, seq_ddict=seq_ddict, task_description={"process_template": {"policy": task_policy_rr}})
                generated_families.append(family)
                gen_task_futures.append(task_fut)
    return asyncio.gather(*gen_task_futures)

async def fold_seq_listen_launch(gen_fam_ddict, fin_fam_ddict, terminate_event):
    processed_families = []
    task_futures = []
    while not terminate_event.is_set():
        families_to_fold = []
        tasks_to_launch = []
        if shared_ddict:
            for family, _ in gen_fam_ddict.items():
                if family not in processed_families:
                    families_to_fold.append(family)
        for family in families_to_fold:
            gpu_alloc = gpu_allocator.next()
            task_policy_rr = Policy(
                placement=Policy.Placement.HOST_NAME,
                host_name=gpu_alloc[0],
                gpu_affinity=gpu_alloc[1],  
            )# async def fold_sequences(generated_families_ddict, superfamily, staging_dir, output_dir, finished_family_ddict, seed=None)
            task_fut = fold_sequence(gen_fam_ddict, family, task_description={"process_template": {"policy": task_policy_rr}})
            processed_families.append(family)
            task_futures.append(task_fut)
    return asyncio.gather(*task_futures)

#async def score_sequence(superfamily, pdb_input_dir, db, tmp_dir, output_file, seq_ddict = None, format_output='query,target,lddt,prob'):
async def score_family_listen_launch(fam_ddict, fin_fam_ddict, seq_ddict=None, terminate_event):
    scored_families = []
    task_futures = []
    while not terminate_event.is_set():
        families_to_score = []
        tasks_to_launch = []
        if shared_ddict:
            for family, _ in fam_ddict.items():
                if family not in scored_families:
                    families_to_score.append(family)
        for family in families_to_score:
            gpu_alloc = gpu_allocator.next()
            task_policy_rr = Policy(
                placement=Policy.Placement.HOST_NAME,
                host_name=gpu_alloc[0],
                gpu_affinity=gpu_alloc[1],  
            )
            task_fut = score_sequence(family, seq_ddict=seq_ddict, task_description={"process_template": {"policy": task_policy_rr}})
            scored_families.append(family)
            task_futures.append(task_fut)
    return asyncio.gather(*task_futures)

_generate_terminate = asyncio.Event()
_folding_terminate = asyncio.Event()
_scoring_terminate = asyncio.Event()
generated_sequences_ddict = {}

superfamily_to_generate_ddict = {}
generated_families_ddict = {}
folded_families_ddict = {}
scored_families_ddict = {}

#gen_seq_listen_launch(superfamily_to_generate_ddict, generated_sequences_ddict, _generate_terminate)
#fold_seq_listen_launch(generated_families_ddict, folded_families_ddict, _folding_terminate)
#score_family_listen_launch(folded_families_ddict, scored_families_ddict, generated_sequences_ddict, _scoring_terminate)